# CRbot: imitation learning v2 (Colab T4)

Trains the `crbot_il_v2` policy / value network on the shards built locally by
`tools/build_il_dataset.py` (VanguardX101/IL_Replay replays run through the CRbot engine).

**Before you run:**
1. Upload the whole `data/il_shards/` folder (all `shard_*.npz` + `manifest.json`, schema `il_shards_v2`) to
   Google Drive as `MyDrive/CRbot_Training/datasets/il_shards/`.
2. On the Mac: `.venv/bin/python tools/pack_ppo_bundle.py --code-only` and upload `dist/crbot_code_bundle.zip`
   to `MyDrive/CRbot_Training/`. The notebook imports the model, the observation layout and the augmentation from
   it, so training always matches the app.
3. Runtime → Change runtime type → **T4 GPU**.

Observation: `il_obs_v2` (`backend_engine/brain/obs_encoder.py`), a **fixed 1752 floats** whatever the size of the
card pool: every card, unit and spell is described by its physical attributes (cost, hp, dps, range, speed,
targeting, splash, ...), never by a card id. Adding cards later changes values, not the model, so this model and
these shards keep working when the roster grows.

Model (`backend_engine/brain/il_model.py`, ~2.9M parameters): a token transformer over a global token, 8 opponent
hand candidates, 5 hand tokens (4 slots + next card), 30 units and 4 spells, with one shared attribute projection.
Four heads: **gate** (wait / play), **card slot** 0..3, **tile** 4 × 576 (masked by the card's legal zone),
**value** (win probability).

Output: `crbot_il_v2.onnx` + `crbot_il_v2.npz` (plain weights for PPO, no pickle) + `crbot_il_v2.json` (obs schema
with `attribute_keys`, metrics, gate threshold) in `MyDrive/CRbot_Training/models/crbot_il_v2/` and as a download.
Put the three files in `CR-Pro-Coach/pretrained_weights/crbot_il_v2/` on the Mac; the app's **Neural (IL v2)**
persona picks them up.

In [ ]:
# 1. Environment
import os, sys, json, math, time, glob, hashlib, random
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !nvidia-smi -L
    !pip -q install onnx onnxruntime onnxscript
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = "/content/drive/MyDrive/CRbot_Training"
    DATA_DIR = f"{ROOT}/datasets/il_shards"
    OUT_DIR = f"{ROOT}/models/crbot_il_v2"
    CODE_DIR = "/content/crbot_code"                # repo code: model, encoder layout, augmentation
    import zipfile
    zipfile.ZipFile(f"{ROOT}/crbot_code_bundle.zip").extractall(CODE_DIR)
else:                                   # local smoke test: IL_SHARDS_DIR / IL_OUT_DIR
    DATA_DIR = os.environ.get("IL_SHARDS_DIR", "data/il_shards")
    OUT_DIR = os.environ.get("IL_OUT_DIR", "il_out")
    CODE_DIR = os.environ.get("CRBOT_REPO", ".")         # run from the repo root
os.makedirs(OUT_DIR, exist_ok=True)
sys.path.insert(0, CODE_DIR)

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print("device:", DEVICE, "| torch", torch.__version__, "| data:", DATA_DIR)

In [ ]:
# 2. Training settings (defaults are tuned for a T4 and ~100k-300k samples)
CFG = dict(
    d_model=256, n_heads=8, n_layers=4, d_ff=512, dropout=0.1,
    batch_size=512, epochs=int(os.environ.get("IL_EPOCHS", 20)), lr=3e-4, weight_decay=0.01, warmup_steps=500,
    w_gate=1.0, w_slot=1.0, w_tile=1.0, w_value=0.5,
    val_fraction=0.1, mirror_aug=True, hand_perm_aug=True,
    max_samples=int(os.environ.get("IL_MAX_SAMPLES", 0)),       # 0 = all (smoke tests use a small number)
)
print(CFG)

In [ ]:
# 3. Load shards into RAM (obs as float16: 1752 x 2 B = ~3.5 KB per sample, ~1.3M samples = ~4.6 GB on a 12.7 GB
#    Colab VM) and split by battle (no battle in both train and val). Arrays are preallocated and filled shard by
#    shard, so there is no concatenate copy. float16 is exact enough: obs values stay within +-5, error < 1e-3.
manifest = json.load(open(f"{DATA_DIR}/manifest.json"))
from backend_engine.brain import obs_encoder as E
assert manifest["schema"] == "il_shards_v2", manifest.get("schema")
assert E.check_model_meta(manifest) is None, E.check_model_meta(manifest)   # shards match this code
OBS_DIM, NUM_TILES = manifest["obs_dim"], manifest["num_tiles"]
print(f"manifest: {manifest['total_samples']:,} samples ({manifest['total_plays']:,} plays) from "
      f"{manifest['total_battles']:,} battles, obs_dim {OBS_DIM} ({len(manifest.get('card_pool', []))} cards "
      f"in the pool when built; the model does not depend on it)")

files = sorted(glob.glob(f"{DATA_DIR}/shard_*.npz"))
assert files, f"no shards in {DATA_DIR}"
keys = ["obs", "gate", "card_slot", "tile_index", "value_target", "slot_mask", "tile_mask", "battle", "side"]
sizes = []
for f in files:
    with np.load(f) as z:
        sizes.append(len(z["gate"]))
N = sum(sizes) if not CFG["max_samples"] else min(sum(sizes), CFG["max_samples"])
try:
    import psutil
    avail = psutil.virtual_memory().available
except ImportError:
    avail = None
need = N * (OBS_DIM * 2 + 72 + 4 + 24)
print(f"loading {N:,} samples: about {need / 2**30:.1f} GB of RAM"
      + (f" ({avail / 2**30:.1f} GB available)" if avail else ""))
assert avail is None or need < 0.8 * avail, "not enough RAM: set IL_MAX_SAMPLES or use a high-RAM runtime"
D, filled = None, 0
for f, n in zip(files, sizes):
    if filled >= N:
        break
    take = min(n, N - filled)
    with np.load(f) as z:
        if D is None:
            D = {k: np.empty((N, *z[k].shape[1:]), np.float16 if k == "obs" else z[k].dtype) for k in keys}
        for k in keys:
            D[k][filled:filled + take] = z[k][:take]
    filled += take
assert D["obs"].shape[1] == OBS_DIM and np.isfinite(D["obs"]).all()

# validation = battles whose id hashes into the last 10% (stable across rebuilds of the same shards)
bucket = np.array([int(hashlib.md5(str(int(b)).encode()).hexdigest(), 16) % 1000 for b in D["battle"]])
is_val = bucket < int(CFG["val_fraction"] * 1000)
tr_idx, va_idx = np.flatnonzero(~is_val), np.flatnonzero(is_val)
print(f"{N:,} samples: train {len(tr_idx):,} / val {len(va_idx):,} "
      f"({len(np.unique(D['battle'][is_val])):,} val battles); play share {D['gate'].mean():.2f}")

T = {
    "obs": torch.from_numpy(D["obs"]),
    "gate": torch.from_numpy(D["gate"].astype(np.float32)),
    "slot": torch.from_numpy(D["card_slot"].astype(np.int64)),
    "tile": torch.from_numpy(D["tile_index"].astype(np.int64)),
    "value": torch.from_numpy(D["value_target"]),
    "slot_mask": torch.from_numpy(D["slot_mask"]),
    "tile_mask": torch.from_numpy(D["tile_mask"]),            # packed uint8 [N, 72], unpacked per batch
}

In [ ]:
# 4. Observation layout + augmentation: imported from the repo (backend_engine/brain/obs_encoder.py, il_model.py)
from backend_engine.brain import il_model as M
GW, GH = E.GRID_W, E.GRID_H
BITS = torch.tensor([128, 64, 32, 16, 8, 4, 2, 1], dtype=torch.uint8)


def unpack_tiles(packed: torch.Tensor) -> torch.Tensor:
    """uint8 [B, 72] -> bool [B, 576] (np.packbits order: most significant bit first)."""
    bits = (packed.unsqueeze(-1) & BITS.to(packed.device)) > 0
    return bits.reshape(packed.shape[0], -1)[:, :NUM_TILES]


def augment(obs, slot, tile, smask, tmask):
    """Left-right mirror (towers swap, x -> 1 - x, tiles mirrored) + hand-slot permutation (il_model.augment)."""
    return M.augment(obs, slot, tile, smask, tmask, mirror=CFG["mirror_aug"], hand_perm=CFG["hand_perm_aug"])

In [ ]:
# 5. Model: backend_engine/brain/il_model.CRbotIL (token transformer, no card-id embedding; exports cleanly to ONNX)
CRbotIL = M.CRbotIL
model = CRbotIL(CFG["d_model"], CFG["n_heads"], CFG["n_layers"], CFG["d_ff"], CFG["dropout"]).to(DEVICE)
print(f"parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M")

In [ ]:
# 6. Losses and validation metrics
NEG = -1e4


def batch_losses(out, gate_t, slot_t, tile_t, value_t, smask, tmask):
    gate, slot, tile, value = out
    play = gate_t > 0.5
    l_gate = F.binary_cross_entropy_with_logits(gate, gate_t)
    l_value = F.binary_cross_entropy(value.clamp(1e-6, 1 - 1e-6), value_t)
    if play.any():
        s_logits = slot[play].masked_fill(~smask[play], NEG)
        l_slot = F.cross_entropy(s_logits, slot_t[play])
        rows = tile[play].gather(1, slot_t[play][:, None, None].expand(-1, 1, NUM_TILES))[:, 0]
        t_logits = rows.masked_fill(~tmask[play], NEG)
        l_tile = F.cross_entropy(t_logits, tile_t[play])
    else:
        l_slot = l_tile = gate.sum() * 0
    total = CFG["w_gate"] * l_gate + CFG["w_slot"] * l_slot + CFG["w_tile"] * l_tile + CFG["w_value"] * l_value
    return total, dict(gate=l_gate.item(), slot=l_slot.item(), tile=l_tile.item(), value=l_value.item())


def get_batch(idx, train):
    b = {k: v[idx].to(DEVICE, non_blocking=True) for k, v in T.items()}
    tmask = unpack_tiles(b["tile_mask"])
    obs, slot, tile, smask = b["obs"].float(), b["slot"], b["tile"], b["slot_mask"]   # obs is stored as float16
    if train:
        obs, slot, tile, smask, tmask = augment(obs, slot, tile, smask, tmask)
    return obs, b["gate"], slot, tile, b["value"], smask, tmask


@torch.no_grad()
def evaluate(idx_all, bs=2048):
    model.eval()
    acc = dict(n=0, gate_ok=0, plays=0, slot_ok=0, top3=0, dist=0.0, brier=0.0, ll=0.0, loss=0.0, batches=0)
    gate_scores, gate_labels = [], []
    for i in range(0, len(idx_all), bs):
        idx = torch.from_numpy(idx_all[i:i + bs])
        obs, gate_t, slot_t, tile_t, value_t, smask, tmask = get_batch(idx, train=False)
        out = model(obs)
        loss, _ = batch_losses(out, gate_t, slot_t, tile_t, value_t, smask, tmask)
        gate, slot, tile, value = out
        acc["loss"] += loss.item(); acc["batches"] += 1
        acc["n"] += len(idx)
        acc["gate_ok"] += ((gate > 0).float() == gate_t).sum().item()
        gate_scores.append(torch.sigmoid(gate).cpu()); gate_labels.append(gate_t.cpu())
        v = value.clamp(1e-6, 1 - 1e-6)
        acc["brier"] += ((v - value_t) ** 2).sum().item()
        acc["ll"] += -(value_t * v.log() + (1 - value_t) * (1 - v).log()).sum().item()
        play = gate_t > 0.5
        if play.any():
            p = play.nonzero()[:, 0]
            acc["plays"] += len(p)
            acc["slot_ok"] += (slot[p].masked_fill(~smask[p], NEG).argmax(1) == slot_t[p]).sum().item()
            rows = tile[p].gather(1, slot_t[p][:, None, None].expand(-1, 1, NUM_TILES))[:, 0]
            rows = rows.masked_fill(~tmask[p], NEG)
            top = rows.topk(3, dim=1).indices
            acc["top3"] += (top == tile_t[p][:, None]).any(1).sum().item()
            pred, true = top[:, 0], tile_t[p]
            dx = (pred % GW - true % GW).float(); dy = (pred // GW - true // GW).float()
            acc["dist"] += torch.sqrt(dx * dx + dy * dy).sum().item()
    n, k = acc["n"], max(acc["plays"], 1)
    scores, labels = torch.cat(gate_scores), torch.cat(gate_labels)
    best_thr, best_f1 = 0.5, -1.0
    for thr in np.arange(0.3, 0.91, 0.05):                       # gate threshold with the best F1 (for NeuralBot)
        pred = scores > thr
        tp = (pred & (labels > 0.5)).sum().item(); fp = (pred & (labels < 0.5)).sum().item()
        fn = (~pred & (labels > 0.5)).sum().item()
        f1 = 2 * tp / max(2 * tp + fp + fn, 1)
        if f1 > best_f1:
            best_thr, best_f1 = float(round(thr, 2)), f1
    model.train()
    return dict(loss=acc["loss"] / max(acc["batches"], 1), gate_acc=acc["gate_ok"] / n,
                slot_top1=acc["slot_ok"] / k, tile_top3=acc["top3"] / k, tile_dist=acc["dist"] / k,
                value_brier=acc["brier"] / n, value_logloss=acc["ll"] / n, gate_threshold=best_thr,
                gate_f1=best_f1)

# baselines on the validation set: always-first-affordable slot, most frequent label tile
va_play = va_idx[D["gate"][va_idx] == 1]
first_ok = D["slot_mask"][va_play].argmax(1)
base_slot = (first_ok == D["card_slot"][va_play]).mean() if len(va_play) else float("nan")
tr_play = tr_idx[D["gate"][tr_idx] == 1]
common_tiles = np.bincount(D["tile_index"][tr_play], minlength=NUM_TILES).argsort()[::-1][:3]
base_top3 = np.isin(D["tile_index"][va_play], common_tiles).mean() if len(va_play) else float("nan")
print(f"baselines (val): slot first-affordable {base_slot:.3f}, tile top-3 most-frequent {base_top3:.3f}")

In [ ]:
# 7. Train (AdamW + warmup / cosine, mixed precision on GPU); keep the best validation loss
opt = torch.optim.AdamW(model.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"])
steps_per_epoch = max(len(tr_idx) // CFG["batch_size"], 1)
total_steps = steps_per_epoch * CFG["epochs"]
sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s + 1) / min(CFG["warmup_steps"], max(total_steps // 10, 1))) *
                                          0.5 * (1 + math.cos(math.pi * min(s / max(total_steps, 1), 1.0))))
use_amp = DEVICE == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
best, history = None, []
ckpt = f"{OUT_DIR}/crbot_il_v2_best.pt"
for epoch in range(CFG["epochs"]):
    t0 = time.time()
    perm = np.random.permutation(tr_idx)
    run = {"gate": 0.0, "slot": 0.0, "tile": 0.0, "value": 0.0}
    for step in range(steps_per_epoch):
        idx = torch.from_numpy(perm[step * CFG["batch_size"]:(step + 1) * CFG["batch_size"]])
        obs, gate_t, slot_t, tile_t, value_t, smask, tmask = get_batch(idx, train=True)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
            out = model(obs)
        out = tuple(o.float() for o in out)
        loss, parts_ = batch_losses(out, gate_t, slot_t, tile_t, value_t, smask, tmask)
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt); scaler.update(); sched.step()
        for k in run:
            run[k] += parts_[k] / steps_per_epoch
    m = evaluate(va_idx)
    history.append({"epoch": epoch + 1, **{f"train_{k}": v for k, v in run.items()}, **m})
    flag = ""
    if best is None or m["loss"] < best["loss"]:
        best = {**m, "epoch": epoch + 1}
        torch.save({"model": model.state_dict(), "cfg": CFG, "metrics": best}, ckpt)
        flag = "  *best*"
    print(f"epoch {epoch + 1:2d} {time.time() - t0:5.0f}s | train g {run['gate']:.3f} s {run['slot']:.3f} "
          f"t {run['tile']:.3f} v {run['value']:.3f} | val loss {m['loss']:.3f} gate {m['gate_acc']:.3f} "
          f"slot {m['slot_top1']:.3f} tile@3 {m['tile_top3']:.3f} dist {m['tile_dist']:.2f} "
          f"brier {m['value_brier']:.3f}{flag}")
print("best:", json.dumps(best, indent=1))

In [ ]:
# 8. Export the best checkpoint: ONNX (inference, verified with onnxruntime) + .npz (plain weights for PPO)
state = torch.load(ckpt, map_location="cpu", weights_only=True)
export_model = CRbotIL(CFG["d_model"], CFG["n_heads"], CFG["n_layers"], CFG["d_ff"], 0.0)
export_model.load_state_dict(state["model"])
export_model.eval()
onnx_path = f"{OUT_DIR}/crbot_il_v2.onnx"
diff = M.export_onnx(export_model, onnx_path, verify=True)
print("onnxruntime vs torch max |diff|:", diff)
M.save_weights(export_model, f"{OUT_DIR}/crbot_il_v2.npz")
assert M.check_against_onnx(M.load_weights(f"{OUT_DIR}/crbot_il_v2.npz"), onnx_path) < 1e-4

meta = M.model_meta(
    "crbot_il_v2",
    gate_threshold=best["gate_threshold"],
    metrics=best,
    baselines={"slot_first_affordable": float(base_slot), "tile_top3_most_frequent": float(base_top3)},
    train={"samples": int(N), "train": int(len(tr_idx)), "val": int(len(va_idx)), "cfg": CFG,
           "dataset": manifest.get("source"), "filters": manifest.get("filters"),
           "card_pool_at_build": manifest.get("card_pool")},
    history=history,
    onnx_check_max_abs_diff=diff,
    onnx_sha256=hashlib.sha256(open(onnx_path, "rb").read()).hexdigest(),
)
json.dump(meta, open(f"{OUT_DIR}/crbot_il_v2.json", "w"), indent=1)
print(f"saved {onnx_path} ({os.path.getsize(onnx_path) / 1e6:.1f} MB) + crbot_il_v2.npz + crbot_il_v2.json")

In [ ]:
# 9. Download (Colab): put the three files in CR-Pro-Coach/pretrained_weights/crbot_il_v2/
if IN_COLAB:
    import shutil
    from google.colab import files
    pack = "/content/crbot_il_v2_pack"                 # .onnx + .npz + .json (no pickled .pt checkpoint)
    os.makedirs(pack, exist_ok=True)
    for name in ("crbot_il_v2.onnx", "crbot_il_v2.npz", "crbot_il_v2.json"):
        shutil.copy(f"{OUT_DIR}/{name}", pack)
    files.download(shutil.make_archive("/content/crbot_il_v2", "zip", pack))